In [1]:
jobtitle="MIS hiring noida"

In [2]:
user_data_dir = r"C:\Users\Chuwi\Desktop\linkdin"

In [3]:
# pip install selenium undetected-chromedriver

In [4]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
import time
import random


profile_directory = "Default"

options = uc.ChromeOptions()
options.add_argument(f"--user-data-dir={user_data_dir}")
options.add_argument(f"--profile-directory={profile_directory}")

options.add_argument("--start-maximized")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
options.add_argument("--disable-gpu")

options.add_argument("--disable-application-cache")
options.add_argument("--disk-cache-size=0")
options.add_argument("--media-cache-size=0")
options.add_argument("--disable-cache")
options.add_argument("--aggressive-cache-discard")
options.add_argument("--start-minimized")
options.add_argument("--disable-backgrounding-occluded-windows")
options.add_argument("--disable-renderer-backgrounding")
options.add_argument("--disable-background-timer-throttling")
driver = uc.Chrome(options=options, version_main=147)

driver.get("https://www.linkedin.com/feed/")
while True:
    if "feed" not in driver.current_url.lower():
        time.sleep(2)
    else:
        break

In [5]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys

In [6]:
search_box = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, 'input[placeholder="Search"]'))
)
time.sleep(3)

In [7]:
search_box.clear()
search_box.send_keys(f"{jobtitle}" + Keys.RETURN)

In [8]:
posts_btn = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.XPATH, '//label[text()="Posts"]'))
)

posts_btn.click()

In [9]:
url = driver.current_url.replace("SWITCH_SEARCH_VERTICAL", "FACETED_SEARCH")

if "datePosted" not in url:
    url += "&datePosted=%5B%22past-24h%22%5D"

driver.get(url)

In [10]:
import time
import random
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException

try:
    main = driver.find_element(By.TAG_NAME, "main")
except NoSuchElementException:
    print("Error: Could not find main element")
    driver.quit()
    exit()

start_time = time.time()
last_data_time = time.time()  # Track when we last found new data
same_count = 0
emails = []
posts = {}
last_height = driver.execute_script("return arguments[0].scrollHeight", main)
no_data_timeout = 30  # Stop if no new data for 30 seconds

while True:
    current_time = time.time()
    
    # Stop after 60 seconds OR if no new data for 30 seconds
    if current_time - start_time > 300:
        print("Stopped: 60 second timeout reached")
        break
    
    if current_time - last_data_time > no_data_timeout:
        print(f"Stopped: No new data for {no_data_timeout} seconds")
        break
    
    try:
        # Scroll
        driver.execute_script(
            "arguments[0].scrollTop += arguments[0].clientHeight * 0.8;",
            main
        )
        time.sleep(0.5)
        
        # Collect data every 2 loops
        if same_count % 2 == 0:
            initial_email_count = len(emails)
            
            # Click "see more" buttons
            try:
                see_more_buttons = driver.find_elements(By.XPATH, "//button[contains(., 'more')]")
                for btn in see_more_buttons:
                    try:
                        driver.execute_script("arguments[0].click();", btn)
                    except:
                        pass
            except:
                pass
            
            # Collect emails
            try:
                links = driver.find_elements(By.TAG_NAME, "a")
                for a in links:
                    try:
                        href = a.get_attribute("href")
                        if href and href.startswith("mailto:") and "@gmail.com" not in href:
                            email = href.replace("mailto:", "")
                            if email not in emails:
                                emails.append(email)
                                try:
                                    parent_div = a.find_element(By.XPATH, "ancestor::p")
                                    div_text = parent_div.text
                                    posts[email] = div_text
                                except:
                                    posts[email] = ""
                    except (StaleElementReferenceException, NoSuchElementException):
                        continue
            except:
                pass
            
            # Update last_data_time if we found new emails
            if len(emails) > initial_email_count:
                last_data_time = current_time
                print(f"Found {len(emails) - initial_email_count} new email(s). Total: {len(emails)}")
        
        # Check scroll height
        new_height = driver.execute_script("return arguments[0].scrollHeight", main)
        if new_height == last_height:
            same_count += 1
            if same_count >= 4:
                print("Stopped: Reached end of page")
                break
        else:
            same_count = 0
            last_height = new_height
            
    except Exception as e:
        print(f"Error during scraping: {e}")
        # Continue trying instead of breaking on errors
        time.sleep(1)
        continue

print(f"\n{'='*50}")
print(f"Total emails collected: {len(emails)}")
print(f"Emails: {list(set(emails))}")  # Remove duplicates
print(f"{'='*50}")

driver.quit()

Found 3 new email(s). Total: 3
Found 1 new email(s). Total: 4
Found 4 new email(s). Total: 8
Found 2 new email(s). Total: 10
Found 4 new email(s). Total: 14
Found 4 new email(s). Total: 18
Found 1 new email(s). Total: 19
Found 5 new email(s). Total: 24
Found 2 new email(s). Total: 26
Found 1 new email(s). Total: 27
Found 2 new email(s). Total: 29
Found 2 new email(s). Total: 31
Found 1 new email(s). Total: 32
Stopped: No new data for 30 seconds

Total emails collected: 32
Emails: ['daniket3.cbsl@tatacapital.com', 'Hr@firstconnectworldwide.com', 'rakshu.rao@tekinspirations.com', 'sharish@comefforts.com', 'md.reza@tatacapital.com', 'info@kandorpeople.com', 'hr@spggcl.com', 'Samiksha.shukla@paisaexpo.com', 'Khyamasagar@lokbharti.ac.in', 'nandini.gandhi@tekinspirations.com', 'sheetal.sharma@dreamjobs.in', 'RESOURCESGLOBALPLACEMENTRGP@GMAIL.COM', 'rahul.kumar@ruloans.com', 'hr@especia.co.in', 'deepika.gahlout@hipp.co.in', 'pallavi.saxena@graygraph.com', 'Pradeep.maurya@lokbharti.ac.in', 'ma

In [11]:
posts

{'hr@f1speedloan.com': '🚨 Walk-in Drive for MIS Executive 🚨\n\nWe Are Hiring!\nJoin our growing team for a Walk-in Drive – MIS Executive\n\n📌 Job Role\nMIS Executive\n\n✅ Eligibility\n• Graduate / BCA / B.Tech / MBA preferred\n• 1–4 years of experience in MIS / Data Management\n• Advanced knowledge of Excel (VLOOKUP, Pivot Table, MIS Reports)\n• Good analytical and communication skills\n• Experience in Fintech / Payday Industry preferred\n\n📍 Walk-in Details\n📅 Date: 14 May 2026\n⏰ Time: 11:00 AM – 4:00 PM\n📍 Venue: Plot No. H144, Sector-63, Noida\n\n📄 Documents to Carry\n• Updated Resume\n• Aadhaar/PAN Copy\n\n🎯 Perks & Benefits\n• Attractive Salary \n• Career Growth Opportunities\n• Supportive Work Environment\n• Immediate Joining\n\n📞 HR Contact: Sneha – 9871989986\n📧 Email: hr@f1speedloan.com\n\nWalk in with your resume and build your career with us!',
 'debanjali.a.nath@capgemini.com': '🚨 Hiring Alert | Capgemini – Noida🚨\n \nCapgemini is hiring PTP & OTC ( Procure To Pay & Order 

In [12]:
# df

# emails

import chatgpt

d=chatgpt.Chatgpt()

final=chatgpt.send(d,posts)

In [13]:
import pandas as pd
df=pd.read_clipboard(sep='|')

df.to_clipboard()

In [13]:


# import re
# import pandas as pd

# rows = []

# for item in final:
#     try:
#         content = re.search(r"\[(.*?)\]", item, re.DOTALL)
#         if content:
#             raw = content.group(1)
    
#             # remove line breaks + extra spaces
#             raw = raw.replace("\n", " ").strip()
    
#             fields = [f.strip() for f in raw.split("|")]
    
#             if len(fields) == 7:
#                 rows.append(fields)
#     except:
#         pass

# df = pd.DataFrame(rows, columns=[
#     "mail id", "company", "location",
#     "experience req", "salary", "skills", "apply"
# ])


# clean_rows = []

# for row in rows:
#     emails = [e.strip() for e in row[0].split(",")]
    
#     for email in emails:
#         new_row = row.copy()
#         new_row[0] = email
#         clean_rows.append(new_row)

# df = pd.DataFrame(clean_rows, columns=[
#     "mail id", "company", "location",
#     "experience req", "salary", "skills", "apply"
# ])


# df.to_csv("linkdin.csv")

# d.quit()

